# Explore reviewed anesthesia EEG

This collaborator template uses GUI dialogs to select a participant HDF5 package and then select one recording inside it. It summarizes metadata and annotations, displays selectable EEG windows, and estimates clean-channel power spectra.

Important: the saved EEG is 0.5–50 Hz filtered and independently z-scored within each channel. Signal values are dimensionless z-scores, and PSD units are **z²/Hz**, not µV²/Hz.

The loader supports both the current single-recording file and a future participant file containing multiple recordings in separate HDF5 groups.

## 1. Imports and file selection

If an import is missing, run `%pip install h5py numpy pandas scipy plotly PyQt6` in a separate cell, restart the kernel, and rerun the notebook.

In [2]:
%pip install pandas

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ------------------------- -------------- 6.3/10.0 MB 38.6 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 34.5 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ----------

In [24]:
from pathlib import Path
import json

import h5py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from scipy.signal import welch
from PyQt6.QtWidgets import QApplication, QFileDialog, QInputDialog

# Open Plotly figures in the normal web browser. This avoids the
# VS Code/Jupyter nbformat MIME-rendering error.
pio.renderers.default = "browser"

# Change this starting folder if the reviewed packages are moved.
DATA_FOLDER = Path(r"C:\Users\zhouz\Downloads\20260626\reorg_data")

# Keep a persistent Qt application for the GUI selection windows.
QT_APP = QApplication.instance() or QApplication([])

selected_package, _ = QFileDialog.getOpenFileName(
    None,
    "Select a participant reviewed EEG package",
    str(DATA_FOLDER),
    "Participant EEG package (*_reviewed_eeg.h5);;HDF5 files (*.h5)",
)

if not selected_package:
    raise RuntimeError("No participant package was selected.")

H5_FILE = Path(selected_package)

print("Opening:", H5_FILE)
print("File size (MB):", round(H5_FILE.stat().st_size / 1_000_000, 2))

Opening: C:\Users\zhouz\Downloads\20260626\reorg_data\L016_reviewed_eeg.h5
File size (MB): 108.2


## 2. Inspect the HDF5 structure

This shows which groups, datasets, dimensions, and attributes are stored without printing the EEG samples.

In [25]:
def print_hdf5_tree(file_path):
    with h5py.File(file_path, "r") as h5_file:
        print("Root attributes:")
        for key, value in h5_file.attrs.items():
            print(f"  {key}: {value}")

        print("\nGroups and datasets:")

        def visitor(name, item):
            indent = "  " * name.count("/")
            if isinstance(item, h5py.Dataset):
                print(
                    f"{indent}- {name} | dataset | "
                    f"shape={item.shape} | dtype={item.dtype}"
                )
            else:
                print(f"{indent}+ {name} | group")

        h5_file.visititems(visitor)

print_hdf5_tree(H5_FILE)

Root attributes:
  format_name: Anesthesia EEG participant package
  n_recordings: 2
  package_created_utc: 2026-07-27T02:20:22.420389+00:00
  package_updated_utc: 2026-07-27T02:24:50.627266+00:00
  participant_id: L016
  schema_version: 2.0

Groups and datasets:
+ recordings | group
  + recordings/or1 | group
    + recordings/or1/annotations | group
      - recordings/or1/annotations/channels | dataset | shape=(10,) | dtype=object
      - recordings/or1/annotations/duration_sec | dataset | shape=(10,) | dtype=float64
      - recordings/or1/annotations/label | dataset | shape=(10,) | dtype=object
      - recordings/or1/annotations/onset_sec | dataset | shape=(10,) | dtype=float64
    + recordings/or1/eeg | group
      - recordings/or1/eeg/channel_labels | dataset | shape=(8,) | dtype=object
      - recordings/or1/eeg/data_zscore | dataset | shape=(8, 4537060) | dtype=float32
      - recordings/or1/eeg/globally_bad_channels | dataset | shape=(5,) | dtype=object
    + recordings/or1/revi

## 3. Load all recordings and embedded metadata

Current files contain one recording at the root. Future participant files may contain `/recordings/<recording_name>/...`. Both layouts are handled here.

In [33]:
def decode_text(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def decode_text_array(dataset):
    return [decode_text(value) for value in dataset[()]]


def attributes_to_dict(attributes):
    result = {}
    for key, value in attributes.items():
        if isinstance(value, np.generic):
            value = value.item()
        result[key] = decode_text(value) if isinstance(value, bytes) else value
    return result


def load_json_dataset(group, name):
    if group is None or name not in group:
        return []
    value = group[name][()]
    return json.loads(decode_text(value))


def load_recording(container, root_attributes, recording_name):
    eeg_group = container["eeg"]
    annotation_group = container.get("annotations")
    review_group = container.get("review")

    metadata = dict(root_attributes)
    metadata.update(attributes_to_dict(container.attrs))
    metadata.update(attributes_to_dict(eeg_group.attrs))

    channel_labels = decode_text_array(eeg_group["channel_labels"])
    globally_bad = (
        decode_text_array(eeg_group["globally_bad_channels"])
        if "globally_bad_channels" in eeg_group
        else []
    )

    if annotation_group is None:
        annotations = pd.DataFrame(
            columns=["onset_sec", "duration_sec", "label", "channels"]
        )
    else:
        annotations = pd.DataFrame(
            {
                "onset_sec": annotation_group["onset_sec"][()].astype(float),
                "duration_sec": annotation_group["duration_sec"][()].astype(float),
                "label": decode_text_array(annotation_group["label"]),
                "channels": decode_text_array(annotation_group["channels"]),
            }
        ).sort_values("onset_sec", ignore_index=True)

    return {
        "name": recording_name,
        "data": eeg_group["data_zscore"][()].astype(np.float32),
        "sampling_rate": float(eeg_group.attrs["sampling_frequency_hz"]),
        "channel_labels": channel_labels,
        "globally_bad_channels": globally_bad,
        "annotations": annotations,
        "metadata": metadata,
        "label_audit": load_json_dataset(review_group, "label_audit_json"),
        "channel_audit": load_json_dataset(review_group, "channel_audit_json"),
        "deleted_annotations": load_json_dataset(
            review_group,
            "deleted_annotations_json",
        ),
    }


with h5py.File(H5_FILE, "r") as h5_file:
    root_attributes = attributes_to_dict(h5_file.attrs)

    if "recordings" in h5_file:
        recording_names = list(h5_file["recordings"].keys())
        if not recording_names:
            raise ValueError("The selected participant package has no recordings.")

        selected_recording, accepted = QInputDialog.getItem(
            None,
            "Select a recording",
            "Recording to inspect:",
            recording_names,
            0,
            False,
        )

        if not accepted or not selected_recording:
            raise RuntimeError("No recording was selected.")

        RECORDING_NAME = str(selected_recording)
        recording = load_recording(
            h5_file["recordings"][RECORDING_NAME],
            root_attributes,
            RECORDING_NAME,
        )
    else:
        RECORDING_NAME = str(
            root_attributes.get("recording_id", H5_FILE.stem)
        )
        recording = load_recording(
            h5_file,
            root_attributes,
            RECORDING_NAME,
        )

# Only the selected recording is loaded into memory.
    recordings = {RECORDING_NAME: recording}

print("Selected package:", H5_FILE)
print("Loaded recording:", RECORDING_NAME)

Selected package: C:\Users\zhouz\Downloads\20260626\reorg_data\L016_reviewed_eeg.h5
Loaded recording: preop1


## 4. Selected recording summary

In [34]:
summary_rows = []

for name, item in recordings.items():
    duration_sec = item["data"].shape[1] / item["sampling_rate"]
    summary_rows.append(
        {
            "recording": name,
            "context": item["metadata"].get("file_context", ""),
            "channels": item["data"].shape[0],
            "samples": item["data"].shape[1],
            "sampling_rate_hz": item["sampling_rate"],
            "duration_min": duration_sec / 60.0,
            "annotations": len(item["annotations"]),
            "globally_bad": ", ".join(item["globally_bad_channels"]),
        }
    )

summary_table = pd.DataFrame(summary_rows)
display(summary_table)

print("Selected recording:", RECORDING_NAME)
print("Channel labels:", recording["channel_labels"])
print("Globally bad channels:", recording["globally_bad_channels"] or "none")

,recording,context,channels,samples,sampling_rate_hz,duration_min,annotations,globally_bad
0,preop1,preop1,8,391920,1000.0,6.532,2,"C3, O1"


Selected recording: preop1
Channel labels: ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
Globally bad channels: ['C3', 'O1']


## 5. Review annotations

`channels = all` means the annotation applies to every channel. Otherwise, the value contains a comma-separated list of affected channels.

In [35]:
annotations = recording["annotations"]

print("Annotation counts by label:")
display(
    annotations["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

display(annotations.head(30))

Annotation counts by label:


,label,count
0,Comment/SSAEP start,1
1,Comment/SSAEP stop,1


,onset_sec,duration_sec,label,channels
0,148.32,0.001,Comment/SSAEP start,all
1,198.06,0.001,Comment/SSAEP stop,all


## 6. Plot a selected EEG window

Change `START_SEC`, `DURATION_SEC`, or `CHANNELS_TO_PLOT`. This is an interactive Plotly figure. The displayed values are limited to ±3 z for readability; the stored EEG is not clipped.

In [36]:
def plot_eeg_window(
    recording_item,
    start_sec=0.0,
    duration_sec=60.0,
    channels=None,
    display_z_limit=3.0,
    channel_spacing=7.0,
    maximum_points=15000,
):
    data = recording_item["data"]
    labels = recording_item["channel_labels"]
    sampling_rate = recording_item["sampling_rate"]

    if channels is None:
        channels = labels

    missing = [channel for channel in channels if channel not in labels]
    if missing:
        raise ValueError(f"Unknown channels: {missing}")

    channel_indices = [labels.index(channel) for channel in channels]
    start_sample = max(0, int(round(start_sec * sampling_rate)))
    end_sample = min(
        data.shape[1],
        int(round((start_sec + duration_sec) * sampling_rate)),
    )
    if end_sample <= start_sample:
        raise ValueError("The requested window is outside the recording.")

    display_step = max(
        1,
        int(np.ceil((end_sample - start_sample) / maximum_points)),
    )
    samples = np.arange(start_sample, end_sample, display_step)
    time_sec = samples / sampling_rate
    offsets = np.arange(len(channels))[::-1] * channel_spacing

    figure = go.Figure()
    colors = [
        "#0072B2", "#D55E00", "#009E73", "#CC79A7",
        "#E69F00", "#56B4E9", "#222222", "#7B2CBF",
    ]

    for row, (channel, channel_index) in enumerate(
        zip(channels, channel_indices)
    ):
        values = np.clip(
            data[channel_index, samples],
            -display_z_limit,
            display_z_limit,
        )
        figure.add_trace(
            go.Scattergl(
                x=time_sec,
                y=values + offsets[row],
                mode="lines",
                name=channel,
                line={"width": 1.2, "color": colors[row % len(colors)]},
                customdata=values,
                hovertemplate=(
                    f"<b>{channel}</b><br>"
                    "Time: %{x:.3f} sec<br>"
                    "Z-score: %{customdata:.2f}<extra></extra>"
                ),
            )
        )

    window_annotations = recording_item["annotations"].loc[
        lambda frame: (
            (frame["onset_sec"] <= end_sample / sampling_rate)
            & (
                frame["onset_sec"] + frame["duration_sec"]
                >= start_sample / sampling_rate
            )
        )
    ]

    top = offsets[0] + display_z_limit
    bottom = offsets[-1] - display_z_limit

    for _, annotation in window_annotations.iterrows():
        onset = float(annotation["onset_sec"])
        duration = float(annotation["duration_sec"])
        label = str(annotation["label"])
        affected = str(annotation["channels"])

        if duration > 0:
            fill = (
                "rgba(255,0,0,0.14)"
                if label.startswith("BAD_")
                else "rgba(30,100,255,0.09)"
            )
            figure.add_vrect(
                x0=onset,
                x1=onset + duration,
                fillcolor=fill,
                line_width=0,
                layer="below",
            )

        figure.add_vline(
            x=onset,
            line_color="red",
            line_dash="dash",
            line_width=1,
        )
        figure.add_annotation(
            x=onset,
            y=top + 0.5,
            text=f"{label} [{affected}]",
            showarrow=False,
            textangle=-45,
            yanchor="top",
            font={"color": "red", "size": 10},
        )

    figure.update_layout(
        title=f"{recording_item['name']}: reviewed EEG",
        xaxis_title="Time from recording start (seconds)",
        yaxis={
            "tickmode": "array",
            "tickvals": offsets,
            "ticktext": channels,
            "range": [bottom - 1, top + 2],
        },
        height=max(650, 100 * len(channels)),
        showlegend=False,
        hovermode="closest",
        plot_bgcolor="white",
        margin={"l": 100, "r": 30, "t": 80, "b": 60},
    )
    figure.update_xaxes(showgrid=True, gridcolor="rgba(180,180,180,0.25)")
    figure.update_yaxes(showgrid=False)
    figure.show(renderer="browser")


START_SEC = 0

# Display the complete recording by default.
DURATION_SEC = (
    recording["data"].shape[1]
    / recording["sampling_rate"]
)
CHANNELS_TO_PLOT = recording["channel_labels"]

plot_eeg_window(
    recording,
    start_sec=START_SEC,
    duration_sec=DURATION_SEC,
    channels=CHANNELS_TO_PLOT,
)

## 7. Calculate clean-data availability

A channel is excluded completely if it is globally bad. Channel-specific `BAD_` annotations remove samples only from the listed channel; `channels = all` removes the interval from every channel.

In [37]:
def bad_sample_mask(recording_item, channel):
    number_samples = recording_item["data"].shape[1]
    sampling_rate = recording_item["sampling_rate"]
    mask = np.zeros(number_samples, dtype=bool)

    if channel in recording_item["globally_bad_channels"]:
        mask[:] = True
        return mask

    for _, annotation in recording_item["annotations"].iterrows():
        label = str(annotation["label"])
        if not label.startswith("BAD_"):
            continue

        channel_text = str(annotation["channels"])
        affected_channels = {
            value.strip()
            for value in channel_text.split(",")
            if value.strip()
        }
        applies_to_channel = (
            channel_text.casefold() == "all"
            or channel in affected_channels
        )
        if not applies_to_channel:
            continue

        start = max(
            0,
            int(np.floor(float(annotation["onset_sec"]) * sampling_rate)),
        )
        end = min(
            number_samples,
            int(
                np.ceil(
                    (
                        float(annotation["onset_sec"])
                        + float(annotation["duration_sec"])
                    )
                    * sampling_rate
                )
            ),
        )
        if end <= start:
            end = min(number_samples, start + 1)
        mask[start:end] = True

    return mask


clean_summary = []
for channel in recording["channel_labels"]:
    mask = bad_sample_mask(recording, channel)
    clean_summary.append(
        {
            "channel": channel,
            "globally_bad": channel in recording["globally_bad_channels"],
            "clean_minutes": (~mask).sum() / recording["sampling_rate"] / 60,
            "clean_percent": 100 * (~mask).mean(),
        }
    )

clean_summary_table = pd.DataFrame(clean_summary)
display(clean_summary_table.round({"clean_minutes": 2, "clean_percent": 1}))

,channel,globally_bad,clean_minutes,clean_percent
0,F3,False,6.53,100.0
1,F4,False,6.53,100.0
2,C3,True,0.00,0.0
3,C4,False,6.53,100.0
4,P3,False,6.53,100.0
5,P4,False,6.53,100.0
6,O1,True,0.00,0.0
7,O2,False,6.53,100.0


## 8. Estimate PSD from completely clean epochs

This divides each channel into fixed-length epochs and uses only epochs that do not overlap a channel-relevant `BAD_` annotation. Globally bad channels are skipped. PSD units are z²/Hz.

In [38]:
def clean_epoch_psd(recording_item, channel, epoch_sec=10.0):
    if channel in recording_item["globally_bad_channels"]:
        return None, None, 0

    labels = recording_item["channel_labels"]
    channel_index = labels.index(channel)
    data = recording_item["data"][channel_index]
    sampling_rate = recording_item["sampling_rate"]
    bad_mask = bad_sample_mask(recording_item, channel)
    epoch_samples = max(2, int(round(epoch_sec * sampling_rate)))

    epoch_psds = []
    frequencies = None

    for start in range(0, len(data) - epoch_samples + 1, epoch_samples):
        end = start + epoch_samples
        if bad_mask[start:end].any():
            continue

        frequencies, power = welch(
            data[start:end],
            fs=sampling_rate,
            nperseg=min(epoch_samples, int(round(4 * sampling_rate))),
            noverlap=None,
            detrend="constant",
            scaling="density",
        )
        epoch_psds.append(power)

    if not epoch_psds:
        return None, None, 0

    return frequencies, np.mean(epoch_psds, axis=0), len(epoch_psds)


psd_figure = go.Figure()
psd_summary = []

for channel in recording["channel_labels"]:
    frequencies, power, number_epochs = clean_epoch_psd(
        recording,
        channel,
        epoch_sec=10.0,
    )
    psd_summary.append(
        {"channel": channel, "clean_10s_epochs": number_epochs}
    )
    if number_epochs == 0:
        continue

    frequency_mask = (frequencies >= 0.5) & (frequencies <= 50.0)
    psd_figure.add_trace(
        go.Scatter(
            x=frequencies[frequency_mask],
            y=10 * np.log10(power[frequency_mask] + np.finfo(float).tiny),
            mode="lines",
            name=channel,
        )
    )

psd_figure.update_layout(
    title=f"{RECORDING_NAME}: mean PSD from clean 10-second epochs",
    xaxis_title="Frequency (Hz)",
    yaxis_title="PSD (dB z²/Hz)",
    template="plotly_white",
)
psd_figure.show(renderer="browser")

display(pd.DataFrame(psd_summary))

,channel,clean_10s_epochs
0,F3,39
1,F4,39
2,C3,0
3,C4,39
4,P3,39
5,P4,39
6,O1,0
7,O2,39


## 9. Review embedded audit information

In [39]:
print("File and processing metadata:")
display(pd.Series(recording["metadata"], name="value").to_frame())

print("Channel audit:")
display(pd.DataFrame(recording["channel_audit"]))

print("Label audit (first 30 rows):")
display(pd.DataFrame(recording["label_audit"]).head(30))

print("Annotations deleted during review (first 30 rows):")
display(pd.DataFrame(recording["deleted_annotations"]).head(30))

File and processing metadata:


,value
format_name,Anesthesia EEG reviewed single-file export
n_recordings,2
package_created_utc,2026-07-27T02:20:22.420389+00:00
package_updated_utc,2026-07-27T02:24:50.627266+00:00
participant_id,L016
schema_version,1.0
file_context,preop1
packaged_utc,2026-07-27T02:20:22.420389+00:00
review_created_utc,2026-07-27T02:19:58.787296+00:00
reviewer,ZZ


Channel audit:


,original_channel,standard_channel,globally_bad,reason
0,F3,F3,False,
1,F4,F4,False,
2,C3,C3,True,
3,C4,C4,False,
4,P3,P3,False,
5,P4,P4,False,
6,O1,O1,True,
7,O2,O2,False,


Label audit (first 30 rows):


,onset_sec,duration_sec,raw_label,standard_label,reviewer_note
0,148.32,0.001,Comment/SSAEP start,Comment/SSAEP start,
1,198.06,0.001,Comment/SSAEP stop,Comment/SSAEP stop,


Annotations deleted during review (first 30 rows):


""


## Recommended future multi-recording layout

Multiple recordings can be stored in one participant HDF5 file, but they should remain separate groups rather than being concatenated. Durations, sampling rates, channel availability, and gaps may differ.

```text
L000_reviewed_eeg.h5
├── participant metadata
└── recordings
    ├── preop1_run-01
    │   ├── eeg
    │   ├── annotations
    │   └── review
    ├── or1_run-02
    │   ├── eeg
    │   ├── annotations
    │   └── review
    └── or2_run-03
        ├── eeg
        ├── annotations
        └── review
```

This notebook will automatically recognize that layout. Keep the original BrainVision recordings and a backup of each reviewed export; one corrupted participant file should not become the only copy of all recordings.